## Building PyG Data Objects
The final result is a Data object that is correct in terms of:
+ Number of nodes
+ Number of edges
+ Correct mask
+ Correct label

##### **Import**

In [8]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data

from src.config import (
    FEATURES_CSV_PATH, CLASSES_CSV_PATH,
    EDGELIST_CSV_PATH, TIME_SPLITS
)

##### **Load all features (165 columns)**

In [10]:
print("Loading full features...")
all_cols = ['txId', 'time_step'] + [f'local_{i}' for i in range(93)] + [f'agg_{i}' for i in range(72)]
df_features = pd.read_csv(FEATURES_CSV_PATH, header=None, names=all_cols)
df_features['txId'] = df_features['txId'].astype('int64')
print(f"Shape: {df_features.shape}")

Loading full features...
Shape: (203769, 167)


##### **Load classes**

In [11]:
df_classes = pd.read_csv(CLASSES_CSV_PATH)
df_classes['txId'] = df_classes['txId'].astype('int64')
df_classes['class'] = df_classes['class'].astype(str)

print(df_classes['class'].value_counts())

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


##### **Create index mapping nodes**
PyG needs index nodes from 0 to N-1. The txId in Elliptic is a random number (~7-8 digits).   
A mapping table needs to be created: txId → index.

In [12]:
all_tx_ids = df_features['txId'].values
tx_to_idx = {tx: idx for idx, tx in enumerate(all_tx_ids)}

print(f"Number of nodes: {len(tx_to_idx):,}")
print(f"Sample mapping: txId {all_tx_ids[0]} → index 0")

Number of nodes: 203,769
Sample mapping: txId 230425980 → index 0


##### **Create a node feature matrix**

In [13]:
feature_cols = [c for c in df_features.columns if c not in ['txId', 'time_step']]
x = torch.tensor(df_features[feature_cols].values, dtype=torch.float)
print(f"x.shape: {x.shape}")

x.shape: torch.Size([203769, 165])


##### **Create `edge_index` from `edgelist`**

In [15]:
print("Loading edgelist...")
df_edges = pd.read_csv(EDGELIST_CSV_PATH)
df_edges.columns = ['txId1', 'txId2']

total_edges = len(df_edges)

mask_valid = (
    df_edges['txId1'].isin(tx_to_idx) &
    df_edges['txId2'].isin(tx_to_idx)
)
df_edges = df_edges[mask_valid].copy()

print(f"Valid edges: {len(df_edges):,} / {total_edges:,}")

src = df_edges['txId1'].map(tx_to_idx).values
dst = df_edges['txId2'].map(tx_to_idx).values
edge_index = torch.tensor(np.array([src, dst]), dtype=torch.long)
print(f"edge_index.shape: {edge_index.shape}")


Loading edgelist...
Valid edges: 234,355 / 234,355
edge_index.shape: torch.Size([2, 234355])


##### **Create label vector**

In [16]:
y = torch.full((len(all_tx_ids),), fill_value=-1, dtype=torch.long)

label_map = {'1': 1, '2': 0}

for _, row in df_classes.iterrows():
    tx_id = row['txId']
    cls   = row['class']
    if tx_id in tx_to_idx and cls in label_map:
        y[tx_to_idx[tx_id]] = label_map[cls]

print(f"licit   (0): {(y == 0).sum().item():,}")
print(f"illicit (1): {(y == 1).sum().item():,}")
print(f"unknown(-1): {(y == -1).sum().item():,}")

licit   (0): 42,019
illicit (1): 4,545
unknown(-1): 157,205


##### **Create `time_step` vector**

In [17]:
time_step = torch.tensor(df_features['time_step'].values, dtype=torch.long)
print(f"time_step range: {time_step.min()}–{time_step.max()}")

time_step range: 1–49


##### **Create `masks`**

In [18]:
train_start, train_end = TIME_SPLITS['train']
val_start,   val_end   = TIME_SPLITS['val']
test_start,  test_end  = TIME_SPLITS['test']

train_mask = (time_step >= train_start) & (time_step <= train_end) & (y != -1)
val_mask   = (time_step >= val_start)   & (time_step <= val_end)   & (y != -1)
test_mask  = (time_step >= test_start)  & (time_step <= test_end)  & (y != -1)

print(f"train_mask: {train_mask.sum().item():,} nodes")
print(f"val_mask:   {val_mask.sum().item():,} nodes")
print(f"test_mask:  {test_mask.sum().item():,} nodes")
print(f"Total masked: {(train_mask | val_mask | test_mask).sum().item():,}")
print(f"Unmasked (unknown): {(~(train_mask | val_mask | test_mask)).sum().item():,}")

train_mask: 26,381 nodes
val_mask:   3,513 nodes
test_mask:  16,670 nodes
Total masked: 46,564
Unmasked (unknown): 157,205


##### **Create PyG Data object**

In [19]:
data = Data(
    x=x,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
)

print(data)

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])
